> **제9회 기출변형 문제** — 교재 476~505쪽
> 문제 476~478쪽 / 정답·해설 479~505쪽
>
> 노트북과 같은 위치에 데이터 파일(csv)을 두고 위에서부터 차례대로 실행하세요.

# 작업형 제1유형 기출동형

## 문제 1. (10점)

제공된 데이터(card_usage.csv)는 지역코드별, 성별에 따른 카드 사용액 데이터이다. 아래 사항을 수행하여 [제출 형식]에 맞춰 답안을 입력하시오.

1. 총사용액은 신용카드사용액(credit_amt)과 체크카드사용액(check_amt)의 합이다.
2. 각 지역코드 내에서, 성별(1: 남자, 2: 여자)별로 총사용액의 합계를 계산하시오.
3. 성별 간 총사용액의 차이(절댓값)가 가장 큰 지역코드를 구하시오.

**제공 데이터 (card_usage.csv)**
| 컬럼명 | 설명 |
|---|---|
| region_code | 지역코드 |
| gender | 성별 (1: 남자, 2: 여자) |
| credit_amt | 신용카드 사용액 |
| check_amt | 체크카드 사용액 |

**[제출 형식]**
- 정수로 작성

### 문제 풀이 방향

지역코드와 성별이라는 두 가지 기준으로 동시에 그룹을 나눠서 합계를 구한 뒤, 같은 지역 안에서 남자 합계와 여자 합계를 나란히 놓고 차이를 계산해야 합니다. 그룹별 합계를 구하는 것까지는 익숙한 흐름이지만, 성별 두 값을 같은 행에 나란히 놓기 위해 표의 모양을 바꿔주는 과정이 이번 문제의 핵심입니다.

### 정답 코드 및 해설

**셀 1) 데이터 불러오기**

In [1]:
import pandas as pd

df = pd.read_csv('card_usage.csv')
print(df.shape)
print(df.head())

(380, 4)
   region_code  gender  credit_amt  check_amt
0            5       1        1802       1289
1            7       1        1598       5859
2            1       1        1344       2315
3            4       1        3425        945
4            8       1        3774       5278


380건의 카드 사용 기록이며, 지역코드는 1~8, 성별은 1(남자)과 2(여자)로 구성되어 있습니다.

**셀 2) 총사용액 컬럼 만들기**

In [2]:
df['total_amt'] = df['credit_amt'] + df['check_amt']
print(df[['region_code', 'gender', 'total_amt']].head())

   region_code  gender  total_amt
0            5       1       3091
1            7       1       7457
2            1       1       3659
3            4       1       4370
4            8       1       9052


신용카드 사용액과 체크카드 사용액을 그대로 더해 총사용액을 만들었습니다. 컬럼끼리 더하는 연산은 같은 행끼리 자동으로 짝지어 계산되므로 반복문이 필요 없습니다.

**셀 3) 지역코드, 성별 두 기준으로 총사용액 합계 구하기**

In [3]:
grp = df.groupby(['region_code', 'gender'])['total_amt'].sum()
print(grp.head(6))

region_code  gender
1            1         155230
             2         134184
2            1         135363
             2         124362
3            1         118896
             2         127968
Name: total_amt, dtype: int64


`groupby(['region_code', 'gender'])`처럼 두 개의 기준을 리스트로 넘기면, 지역코드와 성별이 모두 같은 행끼리 하나의 그룹으로 묶입니다. 그 결과 지역코드 1의 남자 합계, 지역코드 1의 여자 합계, 지역코드 2의 남자 합계처럼 두 기준이 조합된 형태로 결과가 쌓입니다.

**셀 4) 성별을 옆으로 펼쳐서 같은 행에 놓기**

In [4]:
grp_wide = grp.unstack()
print(grp_wide)

gender            1       2
region_code                
1            155230  134184
2            135363  124362
3            118896  127968
4            126301  140999
5            130114  104540
6            140454  126966
7             93829  128601
8            167934  143573


셀 3의 결과는 지역코드와 성별이 세로로 길게 쌓인 형태라서, 같은 지역의 남자 값과 여자 값이 서로 다른 행에 떨어져 있습니다. 이 상태로는 두 값을 바로 빼기 어렵습니다. `unstack()`은 세로로 쌓여 있던 안쪽 그룹 기준(gender)을 컬럼 방향으로 펼쳐주는 함수입니다. 그 결과 지역코드가 하나의 행이 되고, 남자(1)와 여자(2) 값이 같은 행의 서로 다른 컬럼으로 나란히 놓이게 되어, 이제는 컬럼끼리 바로 뺄셈을 할 수 있는 형태가 됩니다.

> 💡 groupby에 기준을 두 개 이상 넘겼을 때, 그 중 하나를 컬럼으로 펼쳐서 나란히 비교하고 싶다면 `unstack()`을 떠올리면 됩니다. 반대로 옆으로 펼쳐진 표를 다시 세로로 쌓고 싶을 때는 `stack()`을 사용합니다.

**셀 5) 성별 간 차이를 구하고 가장 큰 지역코드 찾기 (정답)**

In [5]:
grp_wide['diff'] = (grp_wide[1] - grp_wide[2]).abs()
print(grp_wide)

answer = grp_wide['diff'].idxmax()
print("정답:", answer, "차이:", grp_wide['diff'].max())

gender            1       2   diff
region_code                       
1            155230  134184  21046
2            135363  124362  11001
3            118896  127968   9072
4            126301  140999  14698
5            130114  104540  25574
6            140454  126966  13488
7             93829  128601  34772
8            167934  143573  24361
정답: 7 차이: 34772


`grp_wide[1]`과 `grp_wide[2]`는 각각 남자와 여자의 총사용액 컬럼입니다. 이 둘을 빼고 `.abs()`로 절댓값을 취해 순수한 차이의 크기만 남긴 뒤, `idxmax()`로 그 차이가 가장 큰 행의 인덱스(지역코드)를 찾았습니다.

### 전체 코드

In [6]:
import pandas as pd

df = pd.read_csv('card_usage.csv')
df['total_amt'] = df['credit_amt'] + df['check_amt']

grp_wide = df.groupby(['region_code', 'gender'])['total_amt'].sum().unstack()
grp_wide['diff'] = (grp_wide[1] - grp_wide[2]).abs()

answer = grp_wide['diff'].idxmax()
print("정답:", answer)

정답: 7


### 정리

이 문제의 핵심은 unstack()이었습니다. groupby에 기준을 두 개 넘기면 결과가 세로로 길게 쌓이는데, 그 중 한 기준(성별)을 서로 비교해야 할 때는 옆으로 펼쳐서 같은 행에 놓아야 계산이 편해집니다. 그룹 기준이 하나 늘어날 때마다 결과의 모양이 어떻게 바뀌는지, 그리고 그 모양이 다음 계산에 편한 형태인지를 생각해보는 습관이 필요합니다.

### 🔍 실전 대비: 예상되는 변형과 접근 전략

기출 복원 자료는 응시자의 기억에 의존하기 때문에, 세부 조건(차이인지 비율인지, 어떤 컬럼을 기준으로 삼는지)이 실제와 다르게 전달됐을 가능성이 있습니다. 그래서 원문 그대로를 외우기보다, "이 유형은 결국 무엇을 연습시키려는 것인가"를 파악해두는 것이 더 안전합니다. 이 문제가 연습시키는 핵심은 **두 그룹을 같은 행에 펼쳐두고 비교하는 능력**이며, 비교 방식만 바뀐 변형이 얼마든지 나올 수 있습니다.

**변형 예시: "차이"가 아니라 "배율(비율)"을 구하라고 한다면**

In [9]:
grp_wide['ratio'] = (grp_wide[1] / grp_wide[2]).round(3)
print(grp_wide.sort_values('ratio', ascending=False))
print("배율이 가장 큰 지역:", grp_wide['ratio'].idxmax(), "/ 배율:", grp_wide['ratio'].max())

gender            1       2   diff  ratio
region_code                              
5            130114  104540  25574  1.245
8            167934  143573  24361  1.170
1            155230  134184  21046  1.157
6            140454  126966  13488  1.106
2            135363  124362  11001  1.088
3            118896  127968   9072  0.929
4            126301  140999  14698  0.896
7             93829  128601  34772  0.730
배율이 가장 큰 지역: 5 / 배율: 1.245


grp_wide는 셀 5에서 이미 diff 컬럼(절댓값 차이)을 가지고 있는 상태입니다. 여기에 ratio 컬럼을 새로 추가했기 때문에, 출력된 표에는 diff와 ratio가 나란히 함께 보입니다. 오히려 이렇게 두 지표를 한 표에서 비교해보면, "절대적인 차이"와 "상대적인 배율"이라는 서로 다른 기준이 어떻게 다른 순위를 만들어내는지 더 명확하게 확인할 수 있습니다.

여기서 반드시 짚어야 할 점이 있습니다. "차이가 가장 큰 지역(7번)"과 "배율이 가장 큰 지역(5번)"은 서로 다릅니다. 7번 지역은 절대적인 금액 차이(34,772)는 가장 크지만, 두 그룹 자체의 규모가 커서 상대적인 배율(1.35배 정도)은 5번 지역보다 낮습니다. 반대로 5번 지역은 절대 차이는 작아도 상대적으로는 남성이 여성의 1.245배를 사용해 비율 격차가 더 두드러집니다. 문제 지문에 "차이"라는 단어가 있는지 "비율/배율"이라는 단어가 있는지를 정확히 구분해서 읽는 것이 이런 유형에서 실수를 줄이는 핵심입니다.

---

## 문제 2. (10점)

제공된 데이터(cs_center.csv)는 연도별 고객센터 상담 유형(cs_type)에 대한 접수건수와 처리건수 데이터이다. 아래 사항을 수행하여 [제출 형식]에 맞춰 답안을 입력하시오.

1. 처리율은 처리건수(processed) / 접수건수(received)이다.
2. 각 연도에서 가장 높은 처리율을 기록한 상담 유형을 찾으시오.
3. 해당 상담 유형의 처리건수를 구한 후, 이 값들을 모두(연도별로) 합산하시오.

**제공 데이터 (cs_center.csv)**
| 컬럼명 | 설명 |
|---|---|
| year | 연도 |
| cs_type | 상담 유형 (환불/교환/배송지연/상품문의/품질불량) |
| received | 접수건수 |
| processed | 처리건수 |

**[제출 형식]**
- 정수로 작성

### 문제 풀이 방향

이 데이터는 한 행이 특정 연도의 특정 상담 유형에 대한 접수건수와 처리건수를 담고 있는 형태입니다. 문제가 요구하는 것은 연도마다 그 해에 처리율이 가장 높았던 상담 유형 하나를 뽑고, 그 유형의 처리건수를 확인한 뒤, 8개 연도에서 뽑힌 8개의 처리건수를 모두 더하는 것입니다. 그러므로 먼저 처리율이라는 새 컬럼을 만들고, 연도라는 기준으로 그룹을 나눠서 그룹마다 처리율이 최고인 행을 찾은 다음, 그 행들의 처리건수만 모아서 합산하는 순서로 풀어보겠습니다.

### 정답 코드 및 해설

**셀 1) 데이터 불러오기**

In [10]:
import pandas as pd

df = pd.read_csv('cs_center.csv')
print(df.shape)
print(df.head())

(40, 4)
   year cs_type  received  processed
0  2016      환불       692        387
1  2016      교환       510        356
2  2016    배송지연       452        385
3  2016    상품문의       820        708
4  2016    품질불량      2051       1165


2016년부터 2023년까지 8개 연도, 각 연도마다 5개 상담 유형에 대한 기록이라서 총 40행(8×5)입니다.

**셀 2) 처리율 컬럼 만들기**

In [11]:
df['rate'] = df['processed'] / df['received']
print(df.head())

   year cs_type  received  processed      rate
0  2016      환불       692        387  0.559249
1  2016      교환       510        356  0.698039
2  2016    배송지연       452        385  0.851770
3  2016    상품문의       820        708  0.863415
4  2016    품질불량      2051       1165  0.568016


`processed`와 `received`는 각각 처리건수와 접수건수 컬럼입니다. 두 컬럼을 나누는 연산은 같은 행끼리 자동으로 짝지어 계산되므로, 40행 전체에 대해 반복문 없이 한 번에 처리율이 계산됩니다.

**셀 3) 연도별로 처리율이 가장 높은 행 찾기**

In [12]:
idx = df.groupby('year')['rate'].idxmax()
print(idx)

year
2016     3
2017     7
2018    13
2019    18
2020    23
2021    27
2022    33
2023    38
Name: rate, dtype: int64


`groupby('year')`로 연도가 같은 행들을 하나의 묶음으로 나눕니다. 그리고 그 뒤에 `['rate'].idxmax()`를 붙이면, 각 연도 묶음 안에서 rate 값이 가장 큰 행의 원본 인덱스(위치)를 알려줍니다. 즉 결과로 나온 숫자 3, 7, 13, ...은 df에서 몇 번째 행인지를 나타내는 위치 번호입니다. groupby 없이 df 전체에 대해 `df['rate'].idxmax()`를 했다면 40개 행을 통틀어 처리율이 가장 높은 딱 하나의 행만 찾게 되지만, groupby로 먼저 데이터를 8개의 작은 묶음으로 나눠두면 그 뒤에 붙는 idxmax가 전체가 아니라 각 묶음 안에서만 동작하게 됩니다.

**셀 4) 찾은 행들만 모아서 확인하기**

In [13]:
top = df.loc[idx]
print(top[['year', 'cs_type', 'rate', 'processed']])

    year cs_type      rate  processed
3   2016    상품문의  0.863415        708
7   2017    배송지연  0.871928        674
13  2018    상품문의  0.846721       1149
18  2019    상품문의  0.899441        161
23  2020    상품문의  0.798095        838
27  2021    배송지연  0.899465       1512
33  2022    상품문의  0.829744        809
38  2023    상품문의  0.957090        513


`df.loc[idx]`는 셀 3에서 구한 인덱스 목록(3, 7, 13, ...)에 해당하는 행들만 모아서 보여줍니다. 그 결과 8개 연도 각각에서 처리율이 가장 높았던 상담 유형의 정보가 한 표에 모이게 됩니다. 이 표를 보면 대부분의 해에는 상품문의가, 2017년과 2021년에는 배송지연이 그 해의 최고 처리율 유형이었다는 것을 확인할 수 있습니다.

> 💡 `groupby(기준)[대상컬럼].idxmax()`는 그룹마다 어떤 조건에서 최댓값을 기록한 행이 무엇인지 찾아야 하는 문제에서 자주 쓰이는 패턴입니다. 결과로 나온 인덱스를 `df.loc[]`에 넣으면 그 행들의 다른 컬럼 값도 함께 확인할 수 있습니다.

**셀 5) 처리건수를 모두 합산하기 (정답)**

In [14]:
answer = top['processed'].sum()
print("정답:", answer)

정답: 6364


`top`에 남아있는 것은 8개 행뿐이므로, 이 중 `processed` 컬럼만 골라서 `.sum()`으로 모두 더하면 문제에서 요구하는 최종 합계가 나옵니다.

### 전체 코드

```python
import pandas as pd

df = pd.read_csv('cs_center.csv')
df['rate'] = df['processed'] / df['received']

idx = df.groupby('year')['rate'].idxmax()
top = df.loc[idx]

answer = top['processed'].sum()
print("정답:", answer)

### 정리

이 문제의 핵심은 groupby와 idxmax를 함께 사용하면 그룹별로 조건에 맞는 행을 한 번에 찾을 수 있다는 것입니다. idxmax를 groupby 없이 단독으로 쓰면 데이터 전체에서 딱 하나의 답만 나오지만, groupby로 먼저 연도를 나눠두면 같은 idxmax가 연도마다 각각 적용되어 원하는 개수만큼의 결과가 나옵니다.

### 🔍 실전 대비: 예상되는 변형과 접근 전략

이 문제가 실제로 연습시키는 핵심은 **"그룹별로 조건에 맞는 행을 찾고, 그 행들의 다른 값을 다시 집계하는 2단계 사고"**입니다. "최고"를 "최저"로, "처리건수"를 "접수건수"로 바꾸기만 해도 완전히 다른 문제가 되므로, 이런 방향 전환에 대비해두는 것이 좋습니다.

**변형 예시: "가장 낮은" 처리율을 찾아 "접수건수"를 합산하라고 한다면**

In [15]:
idx = df.groupby('year')['rate'].idxmin()
low = df.loc[idx]
print(low[['year', 'cs_type', 'rate', 'received']])
print("정답(접수건수 합):", low['received'].sum())

    year cs_type      rate  received
0   2016      환불  0.559249       692
5   2017      환불  0.401961       714
10  2018      환불  0.517387       949
15  2019      환불  0.524520       469
20  2020      환불  0.558127      1858
25  2021      환불  0.537736       424
30  2022      환불  0.547762      2324
35  2023      환불  0.551351      1295
정답(접수건수 합): 8725


바뀐 부분은 `idxmax()`가 `idxmin()`으로, `processed`가 `received`로 두 곳뿐입니다. 이 데이터에서는 공교롭게도 8개 연도 모두 "환불" 유형이 가장 처리율이 낮았는데, 실제 시험에서도 이렇게 특정 범주가 매해 반복해서 극값을 차지하는 경우가 흔합니다. 만약 계산 결과가 매년 똑같은 범주로 나온다면, 코드가 잘못된 게 아니라 데이터 자체의 특징일 가능성이 높다는 것도 함께 기억해두면, 시험장에서 "어? 왜 매번 같은 게 나오지?"하며 불필요하게 시간을 낭비하지 않을 수 있습니다.

## 문제 3. (10점)

제공된 데이터(hospital_bill.csv)는 병원 입원 환자 140명의 진료 정보이다. 아래 수행 순서에 따라 데이터를 처리한 후, 구한 값을 [제출 형식]에 맞춰 답안을 입력하시오.

**[수행 순서]**
1. 만족도(`satisfaction`)의 결측치는 전체 만족도의 평균값으로 대체한다.
2. 입원일수(`stay_days`)의 결측치는 같은 진료과(`dept`) 내에서 동일한 중증도(`severity`)를 가진 환자들의 평균 입원일수로 대체한다. 이때, 평균 입원일수는 소수점 이하를 절사하여 정수로 변환하여 사용한다.
3. 각 환자의 `진료비 / 입원일수`(a) 값을 계산하여, 그 값이 세 번째로 큰 환자의 입원일수를 구하시오.
4. 각 환자의 `진료비 / 만족도`(b) 값을 계산하여, 그 값이 두 번째로 큰 환자의 만족도를 구하시오.
5. (3)과 (4)의 값을 더한 결과를 구하시오.

**제공 데이터 (hospital_bill.csv)**
| 컬럼명 | 설명 |
|---|---|
| id | 환자 번호 |
| dept | 진료과 |
| severity | 중증도 (A/B/C) |
| stay_days | 입원일수 |
| bill | 진료비(만원) |
| satisfaction | 만족도 (1~5점) |

**[제출 형식]**
- 반올림하여 소수점 둘째 자리까지 작성

### 문제 풀이 방향

이 문제는 결측치 처리 두 번, 파생변수 생성 두 번, 순위 조회 두 번, 마지막 합산까지 총 다섯 단계로 이어지는 복합 문제입니다. 단계가 많아 보이지만 하나씩 보면 결측치를 채우는 기준(전체 평균 vs 그룹 평균)이 다르다는 것과, 순위를 조회하는 방식이 동일하게 반복된다는 것만 파악하면 어렵지 않습니다. 수행 순서에 적힌 그대로 위에서부터 차례로 처리하면 됩니다.

### 정답 코드 및 해설

**셀 1) 데이터 불러오기**

In [16]:
import pandas as pd

df = pd.read_csv('hospital_bill.csv')
print(df.shape)
print(df.head())

(140, 6)
   id  dept severity  stay_days  bill  satisfaction
0   1    내과        C        3.0   231           1.2
1   2  정형외과        A       11.0   823           4.6
2   3   소아과        A        8.0   339           4.8
3   4  정형외과        B       21.0   631           3.9
4   5    외과        A        3.0   841           1.2


140명의 환자 데이터이며, stay_days와 satisfaction 두 컬럼에 결측치가 섞여 있습니다.

**셀 2) 만족도 결측치를 전체 평균으로 대체**

In [17]:
sat_mean = df['satisfaction'].mean()
df['satisfaction'] = df['satisfaction'].fillna(sat_mean)

print("대체한 평균:", round(sat_mean, 3))
print(df['satisfaction'].isna().sum())

대체한 평균: 2.888
0


`.mean()`은 결측치를 자동으로 제외하고 평균을 계산합니다. `fillna()`는 지정한 값으로 결측치를 채우는 함수이며, 여기서는 방금 구한 전체 평균을 그대로 넣었습니다. 문제에서 전체 평균으로 대체하라고 명시했으므로 조건 없이 데이터 전체를 대상으로 평균을 구한 것입니다. 마지막에 `isna().sum()`으로 결측치가 정말 0개가 되었는지 확인했습니다.

**셀 3) 입원일수 결측치를 그룹 평균으로 대체 (소수점 버림)**

In [18]:
import numpy as np

gmean = df.groupby(['dept', 'severity'])['stay_days'].transform('mean')
df['stay_days'] = df['stay_days'].fillna(gmean)
df['stay_days'] = np.floor(df['stay_days']).astype(int)

print(df['stay_days'].head())
print(df['stay_days'].isna().sum())

0     3
1    11
2     8
3    21
4     3
Name: stay_days, dtype: int32
0


이번에는 전체 평균이 아니라 진료과와 중증도가 모두 같은 환자들끼리의 평균으로 채워야 합니다. `groupby(['dept', 'severity'])`처럼 두 개의 기준을 리스트로 함께 넘기면 두 조건을 모두 만족하는 행끼리 하나의 그룹으로 묶입니다.

`.transform('mean')`을 사용한 이유는 `fillna()`에 곧바로 넣을 수 있는 형태로 결과를 받기 위해서입니다. `groupby(['dept','severity'])['stay_days'].mean()`만 사용하면 그룹별로 압축된 요약표가 나오는데, 이 요약표는 원본 데이터프레임과 행 개수가 다르기 때문에 곧바로 fillna에 넣을 수 없습니다. `transform()`은 그룹 평균을 계산하되 원본과 동일한 행 개수로 다시 펼쳐서 돌려주므로, 각 행에 자신이 속한 그룹의 평균이 그대로 대응되어 fillna에 바로 사용할 수 있습니다.

`np.floor()`를 사용한 이유는 문제에서 절사라는 표현을 썼기 때문입니다. 절사는 반올림하지 않고 소수점 이하를 무조건 잘라낸다는 뜻입니다. `round()`는 소수점 값에 따라 올리거나 내릴 수 있어 이 요구사항에 맞지 않고, `np.floor()`는 항상 소수점 이하를 버리므로 절사의 의미와 정확히 일치합니다.

**셀 4) 파생변수 a, b 생성**

In [19]:
df['a'] = df['bill'] / df['stay_days']
df['b'] = df['bill'] / df['satisfaction']

print(df[['id', 'bill', 'stay_days', 'a', 'satisfaction', 'b']].head())

   id  bill  stay_days           a  satisfaction           b
0   1   231          3   77.000000           1.2  192.500000
1   2   823         11   74.818182           4.6  178.913043
2   3   339          8   42.375000           4.8   70.625000
3   4   631         21   30.047619           3.9  161.794872
4   5   841          3  280.333333           1.2  700.833333


진료비를 입원일수로 나눈 a는 입원 1일당 진료비를, 진료비를 만족도로 나눈 b는 만족도 1점당 진료비를 의미합니다. 절대적인 진료비 크기만으로는 비교하기 어려운 두 환자를 비교할 때, 다른 값으로 나누어 비율로 바꾸면 상대적인 크기를 비교할 수 있습니다.

**셀 5) a값 3번째로 큰 환자의 입원일수 (답3)**

In [20]:
answer3 = df.sort_values('a', ascending=False).iloc[2]['stay_days']
print("답3:", answer3)

답3: 1


`sort_values('a', ascending=False)`는 a값을 기준으로 큰 값부터 작은 값 순서로 데이터프레임 전체를 재배열합니다. 정렬한 뒤에는 위에서부터 순서대로 1등, 2등, 3등이 되므로 `.iloc[0]`이 1등, `.iloc[1]`이 2등, `.iloc[2]`가 3등에 해당합니다. `.iloc[2]`로 3등 행 전체를 가져온 뒤 `['stay_days']`로 그 환자의 입원일수만 꺼냅니다.

> 💡 `sort_values(기준, ascending=False).iloc[n-1]`은 n등 값을 찾는 문제에서 그대로 재사용할 수 있는 패턴입니다. 1등이면 iloc[0], 5등이면 iloc[4]를 사용하면 됩니다.

**셀 6) b값 2번째로 큰 환자의 만족도 (답4)**

In [21]:
answer4 = df.sort_values('b', ascending=False).iloc[1]['satisfaction']
print("답4:", answer4)

답4: 1.3


셀 5와 같은 방식이며, 이번에는 2등을 찾아야 하므로 `.iloc[1]`을 사용합니다.

**셀 7) 최종 정답 (답3 + 답4)**

In [22]:
final_answer = round(answer3 + answer4, 2)
print("최종 정답:", final_answer)

최종 정답: 2.3


앞서 구한 답3(정수형 입원일수)과 답4(소수형 만족도)를 단순히 더하고, 문제에서 요구한 대로 소수 둘째 자리까지 반올림합니다.

### 전체 코드

In [23]:
import pandas as pd
import numpy as np

df = pd.read_csv('hospital_bill.csv')

# 1) 만족도 결측치 -> 전체 평균
df['satisfaction'] = df['satisfaction'].fillna(df['satisfaction'].mean())

# 2) 입원일수 결측치 -> (dept, severity) 그룹 평균, 절사하여 정수화
gmean = df.groupby(['dept', 'severity'])['stay_days'].transform('mean')
df['stay_days'] = df['stay_days'].fillna(gmean)
df['stay_days'] = np.floor(df['stay_days']).astype(int)

# 3), 4) 파생변수 생성
df['a'] = df['bill'] / df['stay_days']
df['b'] = df['bill'] / df['satisfaction']

# 5) 순위 조회 및 합산
answer3 = df.sort_values('a', ascending=False).iloc[2]['stay_days']
answer4 = df.sort_values('b', ascending=False).iloc[1]['satisfaction']

final_answer = round(answer3 + answer4, 2)
print("답3:", answer3)
print("답4:", answer4)
print("최종 정답:", final_answer)

답3: 1
답4: 1.3
최종 정답: 2.3


### 정리

이 문제는 결측치 처리 방식이 컬럼마다 다르게 지시된다는 점이 핵심입니다. satisfaction은 조건 없는 전체 평균으로, stay_days는 (dept, severity)라는 두 가지 조건을 모두 만족하는 그룹 평균으로 채웠습니다. 이후 a, b라는 두 개의 비율 변수를 만들고 각각 정렬해서 원하는 등수의 값을 꺼내는 과정은 동일한 패턴(sort_values + iloc)의 반복이었습니다.

### 🔍 실전 대비: 예상되는 변형과 접근 전략

이 5단계 구조(결측치 2번 + 파생변수 2번 + 순위 조회 2번 + 합산)를 처음 보면 다소 인위적으로 느껴질 수 있습니다. 하지만 실제로 이 조합이 반복 출제되는 이유는, **결측치 처리 방식 두 가지(전체 평균 vs 그룹 평균)와 비율 변수 해석 능력을 한 문제 안에서 동시에 평가할 수 있기 때문**입니다. 즉 문제 하나로 여러 개념을 확인할 수 있어 출제자 입장에서 효율적인 구성이며, 이 문제 자체가 특별히 부자연스럽게 설계된 것은 아니라고 판단했습니다. 다만 세부 조건(몇 번째 큰 값을 찾는지, 절편·버림 방식이 무엇인지)은 회차마다 조금씩 바뀔 수 있으므로, "구조"를 외우기보다 "단계별로 무엇을 하는지"를 이해해두는 것이 중요합니다.

**변형 예시: "가장 큰" 값이 아니라 "가장 작은" 값을 찾으라고 한다면**

In [24]:
s = df.sort_values('a', ascending=True)
print(s[['id', 'bill', 'stay_days', 'a']].head(5))
print("a값이 가장 작은 환자의 진료비:", s.iloc[0]['bill'])

      id  bill  stay_days         a
25    26   106         21  5.047619
49    50   108         18  6.000000
135  136   152         22  6.909091
28    29   185         24  7.708333
83    84    85         11  7.727273
a값이 가장 작은 환자의 진료비: 106


`sort_values`의 `ascending` 옵션 하나만 `True`로 바꾸면 "가장 큰 값"을 찾던 코드가 그대로 "가장 작은 값"을 찾는 코드가 됩니다. a값(진료비/입원일수)이 작다는 것은 입원일수 대비 진료비가 낮다는 뜻으로, 문제에서 "가장 비효율적인" 또는 "가장 저비용인" 환자처럼 표현을 바꿔 물어볼 수도 있습니다. 이렇게 지문의 표현이 "가장 크다/많다"인지 "가장 작다/적다"인지를 놓치지 않고 `ascending` 옵션에 정확히 반영하는 것이 이 유형에서 실수를 막는 핵심입니다.

---

## 이번 회차에서 사용한 핵심 함수 정리
| 상황 | 함수 | 핵심 이유 |
|---|---|---|
| 두 개 이상의 기준으로 동시에 그룹 나누기 | `groupby([기준1, 기준2])` | 두 조건을 모두 만족하는 행끼리 묶을 때 |
| 세로로 쌓인 그룹 결과를 옆으로 펼치기 | `unstack()` | 같은 행에서 서로 다른 그룹 값끼리 비교/연산하고 싶을 때 |
| 그룹별로 조건에 맞는 행(최댓값을 가진 행)을 각각 찾기 | `groupby(기준)[대상].idxmax()` + `df.loc[인덱스]` | groupby로 나눈 뒤에는 뒤따르는 함수가 각 그룹 안에서 독립적으로 동작 |
| 그룹별 평균을 원본 행 개수에 맞춰 계산 | `groupby().transform('mean')` | 그룹 요약표가 아니라 원본과 동일한 행 개수로 반환되어 fillna에 바로 사용 가능 |
| 소수점 버림(절사) | `np.floor()` | round와 달리 항상 아래로 버려서 절사의 의미와 일치 |
| n등 값 찾기 | `sort_values(ascending=False).iloc[n-1]` | 정렬 순서와 인덱스 번호를 이용해 등수를 바로 조회 |

---

# 작업형 2유형

## 문제

제공된 학습용 데이터(train.csv)는 토양 성분 및 기후 조건에 따른 작물 재배 기록 데이터이다. 학습용 데이터를 활용하여 가장 적합한 작물 유형(crop_type)을 예측하는 모형을 개발하고, 모형을 평가용 데이터(test.csv)에 적용하여 작물 유형을 예측하시오. 예측 결과는 아래의 [유의사항]과 [제출 형식]을 준수하여, CSV 파일(result.csv)을 생성하는 코드를 제출하시오.

**[유의사항]**
① 예측 결과는 F1 Score(macro) 평가지표에 따라 평가함
② 타겟 변수(crop_type)는 0, 1, 2 총 3개의 클래스로 구성됨 (0: 곡물, 1: 채소, 2: 과수)

**[제출 형식]**
① CSV 파일명: result.csv
② 예측 작물유형 컬럼명: pred
③ 제출 컬럼 개수: pred 컬럼 1개
④ 평가용 데이터 개수와 예측 결과 데이터 개수 일치: 900개

**[제공 데이터]**
- train.csv: 학습용 데이터, 약 2,600개
- test.csv: 평가용 데이터, 약 900개 (crop_type 컬럼 미제공)

| 컬럼명 | 설명 |
|---|---|
| id | 재배 구역 ID |
| nitrogen | 토양 질소 함량 |
| phosphorus | 토양 인산 함량 |
| potassium | 토양 칼륨 함량 |
| soil_ph | 토양 산도(pH) |
| temperature | 평균 기온(℃) |
| humidity | 평균 습도(%) |
| rainfall | 강수량(mm) |
| crop_type | (train에만 존재) 작물유형 — 0: 곡물, 1: 채소, 2: 과수 |

### 문제 풀이 방향

2유형은 정답 하나를 구하는 것이 아니라, 학습 → 예측 → 제출파일 생성까지 이어지는 전체 파이프라인을 코드로 완성하는 것이 목표입니다. 이 문제는 토양·기후 특성으로 3개 클래스 중 하나를 맞히는 다중분류이므로, 순서는 다음과 같습니다.

1. 데이터를 불러와 구조와 결측치, 클래스 비율을 확인한다
2. 결측치를 처리한다
3. 학습 데이터 일부를 떼어내 검증용으로 사용하여, 실제 채점 전에 모델의 성능을 미리 가늠해본다
4. 모델을 학습시키고 검증 데이터로 F1-macro 점수를 확인한다
5. 확인이 끝나면 전체 학습 데이터로 다시 학습시킨 뒤, 진짜 평가 데이터(test.csv)를 예측한다
6. 요구된 형식(pred 컬럼 1개, 900행)으로 result.csv를 저장한다

이번 데이터는 모든 독립변수가 이미 숫자형이라는 점이 특징입니다. 그래서 범주형 인코딩 단계 없이 결측치 처리 이후 바로 모델링으로 넘어갈 수 있습니다.

### 정답 코드 및 해설

**셀 1) 데이터 불러오기 + 구조 확인**

In [26]:
import pandas as pd

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(train.shape, test.shape)
train.head()

(2600, 9) (900, 8)


,id,nitrogen,phosphorus,potassium,soil_ph,temperature,humidity,rainfall,crop_type
0,2730,52,114,106,6.59,23.9,79.2,186.9,1
1,2009,53,118,94,5.98,31.9,98.1,106.8,1
2,841,112,34,60,NaN,29.6,60.4,370.8,0
3,2635,70,112,128,5.91,31.8,54.8,219.8,1
4,1722,66,98,192,NaN,16.7,48.5,131.3,0


train은 9개 컬럼(crop_type 포함), test는 8개 컬럼(crop_type 없음)입니다. 첫 5행만 봐도 soil_ph에 결측치(NaN)가 섞여 있는 것이 눈에 띕니다. 이런 결측치는 다음 셀에서 정확한 개수를 확인하고 처리하겠습니다.

**셀 2) 결측치 및 타겟 분포 확인**

In [27]:
print(train.isna().sum())
print(test.isna().sum())
print(train['crop_type'].value_counts())

id               0
nitrogen         0
phosphorus       0
potassium        0
soil_ph        101
temperature      0
humidity        67
rainfall         0
crop_type        0
dtype: int64
id              0
nitrogen        0
phosphorus      0
potassium       0
soil_ph        29
temperature     0
humidity       23
rainfall        0
dtype: int64
crop_type
0    1160
1     873
2     567
Name: count, dtype: int64


soil_ph와 humidity 두 컬럼에 결측치가 있습니다. `value_counts()`로 확인한 crop_type 분포를 보면 곡물(0)이 가장 많고 과수(2)가 가장 적어, 세 클래스의 비율이 완전히 균등하지는 않다는 것도 알 수 있습니다. 이렇게 클래스 비율이 고르지 않은 상태를 미리 파악해두면, 이후 데이터를 나눌 때 `stratify` 옵션이 왜 필요한지 이해하는 데 도움이 됩니다.

**셀 3) 결측치 처리**

In [28]:
for col in ['soil_ph', 'humidity']:
    med = train[col].median()
    train[col] = train[col].fillna(med)
    test[col] = test[col].fillna(med)

print(train.isna().sum().sum(), test.isna().sum().sum())

0 0


soil_ph와 humidity 모두 train에서 계산한 중앙값으로 train과 test 양쪽에 동일하게 채웠습니다. 두 컬럼을 한 번에 처리하기 위해 `for col in ['soil_ph', 'humidity']`로 반복문을 사용했는데, 처리해야 할 컬럼이 여러 개일 때 각각 따로 코드를 쓰는 대신 반복문으로 묶으면 코드가 짧아지고 실수도 줄어듭니다. 마지막 줄에서 `isna().sum().sum()`으로 train과 test 전체에 결측치가 하나도 남지 않았는지 확인했습니다.

> 💡 train의 중앙값을 test에도 그대로 적용해야 한다는 원칙은 결측치를 채울 때뿐 아니라, 인코딩 규칙이나 스케일링 기준을 정할 때도 동일하게 적용됩니다. "test 정보를 미리 들여다보지 않는다"는 원칙을 항상 기억해두면 좋습니다.

**셀 4) 학습용/검증용 데이터 분리**

In [29]:
from sklearn.model_selection import train_test_split

X = train.drop(columns=['id', 'crop_type'])
y = train['crop_type']
X_test = test.drop(columns=['id'])

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(X_tr.shape, X_val.shape)

(2080, 7) (520, 7)


`id`는 재배 구역을 구분하기 위한 값일 뿐 작물 유형과 관련이 없으므로 학습에 사용할 컬럼(X)에서 제외했습니다. 셀 2에서 확인했듯 crop_type의 세 클래스 비율이 고르지 않으므로, `stratify=y`를 지정해 학습용과 검증용 양쪽에 원본과 동일한 클래스 비율이 유지되도록 나눴습니다.

**셀 5) 모델 학습 및 검증 성능 확인**

In [30]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

model = RandomForestClassifier(n_estimators=300, random_state=42)
model.fit(X_tr, y_tr)

val_pred = model.predict(X_val)
print("검증 F1-macro:", f1_score(y_val, val_pred, average='macro'))

검증 F1-macro: 0.720854472760687


`RandomForestClassifier`는 여러 개의 결정나무를 만들고 그 예측을 종합해서 최종 판단을 내리는 모델로, 특별한 전처리 없이도 안정적인 성능을 내는 경우가 많아 2유형에서 기본으로 시도하기 좋습니다. `f1_score(..., average='macro')`는 세 클래스(곡물/채소/과수) 각각에 대해 F1 점수를 따로 구한 뒤 단순 평균 낸 값입니다. 클래스 개수가 상대적으로 적은 과수(2)를 잘 맞히는지도 곡물(0)만큼 동등하게 반영된다는 뜻입니다.

**셀 6) 전체 학습 데이터로 재학습 후 test 예측**

In [31]:
model.fit(X, y)
pred = model.predict(X_test)

print(pred[:10])

[1 1 0 1 0 0 0 2 1 1]


셀 5에서는 검증을 위해 train 데이터의 80%만 사용해서 학습했습니다. 성능을 확인했으니, 이제는 가지고 있는 train 데이터를 전부 사용해서 다시 학습시킵니다. 이렇게 재학습한 모델로 test 데이터를 예측한 결과, 각 재배 구역이 0(곡물), 1(채소), 2(과수) 중 어디에 해당하는지가 배열로 나왔습니다.

**셀 7) 제출 파일 생성**

In [32]:
result = pd.DataFrame({'pred': pred})
result.to_csv('result.csv', index=False)

print(result.shape)
print(result.head())

(900, 1)
   pred
0     1
1     1
2     0
3     1
4     0


제출 형식에서 요구한 대로 `pred`라는 컬럼 하나만 있는 데이터프레임을 만들고 저장했습니다. `index=False`를 빠뜨리면 행 번호가 별도 컬럼으로 함께 저장되어 pred 컬럼 1개라는 조건을 어기게 되므로 반드시 챙겨야 합니다. 저장 후 `result.shape`으로 행 개수가 test 데이터 개수(900개)와 정확히 일치하는지 확인했습니다.

### 전체 코드

In [33]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# 결측치 처리 (train 기준으로 계산한 값을 test에도 동일 적용)
for col in ['soil_ph', 'humidity']:
    med = train[col].median()
    train[col] = train[col].fillna(med)
    test[col] = test[col].fillna(med)

X = train.drop(columns=['id', 'crop_type'])
y = train['crop_type']
X_test = test.drop(columns=['id'])

# 검증용 분리 후 성능 확인
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=300, random_state=42)
model.fit(X_tr, y_tr)
val_pred = model.predict(X_val)
print("검증 F1-macro:", f1_score(y_val, val_pred, average='macro'))

# 전체 데이터로 재학습 후 test 예측
model.fit(X, y)
pred = model.predict(X_test)

# 제출 파일 생성
result = pd.DataFrame({'pred': pred})
result.to_csv('result.csv', index=False)
print(result.shape)

검증 F1-macro: 0.720854472760687
(900, 1)


### 정리

이 문제는 모든 독립변수가 이미 숫자형이라 범주형 인코딩이 필요 없었던 만큼, 결측치 처리 → 검증 분리 → 학습 → 예측 → 제출이라는 2유형의 기본 골격에 더 집중해서 연습할 수 있는 문제였습니다. 클래스 비율이 균등하지 않을 때 `stratify`로 분리하고 `f1_score`를 `average='macro'`로 평가해야 하는 이유를 다시 한번 확인했습니다.

이번 모델의 검증 F1-macro는 약 0.72 수준이었습니다.

### 🔍 실전 대비: 예상되는 변형과 접근 전략

기출 복원 자료에는 "작물 유형 다중분류, F1-macro"라는 큰 틀만 전해지고, 정확히 어떤 독립변수가 주어졌는지까지는 세세하게 전달되지 않는 경우가 많습니다. 그래서 실제 시험에서는 컬럼 이름이나 개수가 이 문제와 다르게 나올 수 있습니다. 다만 "숫자형 변수 위주의 다중분류, 결측치 처리 후 F1-macro로 평가"라는 뼈대는 유지될 가능성이 높으므로, 이 흐름 자체를 손에 익혀두는 것이 중요합니다.

**변형 예시: 결측치가 있는 컬럼이 늘어나거나, 범주형 변수가 섞여 있다면**

실전에서는 결측치가 있는 컬럼이 이번 문제(soil_ph, humidity)보다 많을 수도 있고, 반대로 전혀 없을 수도 있습니다. 또한 이 문제와 달리 문자열로 된 범주형 변수(예: 재배 지역, 토양 종류)가 섞여 있다면, 결측치 처리 다음 단계로 `LabelEncoder`를 이용한 인코딩이 추가되어야 합니다. 아래 코드는 앞서 다룬 셀 3(결측치 처리)을 대신할 수 있는 방법으로, 데이터를 다시 불러온 상태에서 시작합니다.

In [34]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# 결측치가 있는 컬럼을 자동으로 찾아서 한 번에 처리하는 방법
missing_cols = train.columns[train.isna().sum() > 0].tolist()
print("결측치가 있는 컬럼:", missing_cols)

for col in missing_cols:
    med = train[col].median()
    train[col] = train[col].fillna(med)
    test[col] = test[col].fillna(med)

print(train.isna().sum().sum())

결측치가 있는 컬럼: ['soil_ph', 'humidity']
0


`train.columns[train.isna().sum() > 0]`는 각 컬럼의 결측치 개수를 센 뒤, 0보다 큰(즉 결측치가 있는) 컬럼의 이름만 골라줍니다. 이렇게 결측치가 있는 컬럼을 코드로 자동 탐지해두면, 실제 시험에서 결측치 컬럼의 개수나 이름이 이 문제와 다르게 나오더라도 컬럼명을 일일이 눈으로 확인해서 하드코딩할 필요 없이 동일한 코드로 대응할 수 있습니다. 시험 시간이 촉박할 때 특히 유용한 습관입니다.

## 이번 회차에서 사용한 핵심 개념 정리
| 상황 | 함수/개념 | 핵심 이유 |
|---|---|---|
| 여러 컬럼의 결측치를 한 번에 처리 | `for col in [컬럼리스트]: ...` | 컬럼마다 반복되는 코드를 줄이고 실수 방지 |
| 결측치가 있는 컬럼을 자동으로 탐지 | `train.columns[train.isna().sum() > 0]` | 컬럼명을 미리 알지 못해도 코드로 찾아낼 수 있음 |
| 클래스 불균형 상태로 데이터 분리 | `stratify=y` | 학습용/검증용에 클래스 비율을 동일하게 유지 |
| 다중분류 평가 | `f1_score(..., average='macro')` | 클래스별 F1을 동일 가중치로 평균 (소수 클래스도 동등하게 반영) |
| 제출 파일 저장 | `to_csv('result.csv', index=False)` | index=False 없으면 불필요한 인덱스 컬럼이 추가됨 |

---

# 작업형 3유형

## 문제 1. (10점)

제공된 데이터(process_quality.csv)는 제조 공정의 조건과 제품 품질점수에 관한 자료이다. id를 기준으로 train(1~140)과 test(141~200) 데이터로 나누어, train 데이터로 다중선형 회귀분석을 수행하여 품질점수(quality_score)를 예측하고, test 데이터로 모형의 성능을 평가하시오.

**제공 데이터 (process_quality.csv)**
| 컬럼명 | 설명 |
|---|---|
| id | 생산 배치 번호 |
| temp | 공정 온도(℃) |
| pressure | 공정 압력(bar) |
| speed | 생산 속도(m/min) |
| humidity | 작업장 습도(%) |
| quality_score | (종속변수) 제품 품질점수 |

**문제 1-1** train 데이터에서 id를 제외한 모든 변수를 독립변수로 사용하여 다중선형 회귀모형을 적합하시오. 절편을 제외한 독립변수 중 유의확률(p-value)이 0.05보다 작은 변수의 개수를 구하시오.

**문제 1-2** 1-1에서 유의한 변수만 사용하여 다중선형 회귀모형을 다시 적합하시오. train 데이터에서 이 모형의 예측값과 실제 quality_score 값 사이의 피어슨 상관계수를 구하시오. (반올림하여 소수 셋째 자리까지 작성)

**문제 1-3** 1-2에서 적합한 회귀모형을 이용하여 test 데이터에서 quality_score의 예측값을 산출한 후, 예측값과 실제 quality_score 값 사이의 RMSE(Root Mean Squared Error)를 구하시오. (반올림하여 소수 셋째 자리까지 작성)

### 문제 풀이 방향

이 문제는 다중선형 회귀를 train/test로 나누어 다루면서, 두 가지 성능 확인 방법을 함께 연습합니다. 하나는 모형을 학습시킨 train 데이터 자체에서 예측값과 실제값이 얼마나 비슷한 패턴으로 움직이는지를 피어슨 상관계수로 확인하는 것이고, 다른 하나는 모형이 한 번도 보지 못한 test 데이터에 적용했을 때 예측이 실제와 얼마나 차이 나는지를 RMSE로 확인하는 것입니다.

### 정답 코드 및 해설

**셀 1) 데이터 불러오기**

In [35]:
import pandas as pd

df = pd.read_csv('process_quality.csv')
print(df.shape)
df.head()

(200, 6)


,id,temp,pressure,speed,humidity,quality_score
0,1,184.2,6.25,130.3,58.8,88.02
1,2,176.8,3.34,71.5,62.7,71.74
2,3,203.8,9.42,58.8,62.5,70.40
3,4,227.9,7.33,75.5,41.5,77.64
4,5,233.7,3.19,93.8,49.2,98.14


200개 생산 배치 데이터이며, 모든 컬럼이 숫자형이라 별도의 범주형 인코딩은 필요하지 않습니다.

**셀 2) id 기준 train/test 분할**

In [36]:
train = df[df['id'] <= 140].reset_index(drop=True)
test = df[df['id'] > 140].reset_index(drop=True)

print("train:", train.shape, "test:", test.shape)

train: (140, 6) test: (60, 6)


id가 140 이하인 140개 배치는 train으로, 141 이상인 60개는 test로 나눴습니다.

**셀 3) train 데이터로 다중선형 회귀모형 적합**

In [37]:
import statsmodels.api as sm

X_train = train.drop(columns=['id', 'quality_score'])
y_train = train['quality_score']
X_train = sm.add_constant(X_train)

model = sm.OLS(y_train, X_train).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:          quality_score   R-squared:                       0.818
Model:                            OLS   Adj. R-squared:                  0.813
Method:                 Least Squares   F-statistic:                     151.9
Date:                Sun, 19 Jul 2026   Prob (F-statistic):           5.96e-49
Time:                        22:29:07   Log-Likelihood:                -435.18
No. Observations:                 140   AIC:                             880.4
Df Residuals:                     135   BIC:                             895.1
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         39.7578      4.177      9.519      0.0

R-squared가 0.818로, 이 모형이 품질점수 변동의 약 81.8%를 설명하고 있습니다. 계수표를 보면 temp, pressure, speed는 p-value가 0.05보다 훨씬 작아 유의하지만, humidity는 p-value가 0.676으로 전혀 유의하지 않습니다. 즉 이 데이터 안에서는 습도가 품질점수를 설명하는 데 통계적으로 의미 있는 역할을 하지 못하는 것으로 나타났습니다.

**셀 4) 유의한 변수 개수 확인 (문제 1-1 정답)**

In [38]:
sig_vars = model.pvalues[model.pvalues < 0.05].index.tolist()
sig_count = len(sig_vars) - 1  # 절편(const) 제외

print("유의한 변수:", sig_vars)
print("1-1 정답:", sig_count)

유의한 변수: ['const', 'temp', 'pressure', 'speed']
1-1 정답: 3


`model.pvalues < 0.05`로 유의한 변수를 걸러낸 뒤, 절편(const)을 제외한 개수를 셌습니다.

**셀 5) 유의한 변수만으로 재적합**

In [39]:
X_train2 = X_train[sig_vars]
model2 = sm.OLS(y_train, X_train2).fit()
print(model2.summary())

                            OLS Regression Results                            
Dep. Variable:          quality_score   R-squared:                       0.818
Model:                            OLS   Adj. R-squared:                  0.814
Method:                 Least Squares   F-statistic:                     203.7
Date:                Sun, 19 Jul 2026   Prob (F-statistic):           4.18e-50
Time:                        22:29:40   Log-Likelihood:                -435.27
No. Observations:                 140   AIC:                             878.5
Df Residuals:                     136   BIC:                             890.3
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         39.0668      3.824     10.215      0.0

humidity를 제거하고 재적합한 결과, 남은 세 변수 모두 p-value 0.000으로 뚜렷하게 유의합니다. R-squared는 0.818로 원래 모형과 거의 동일하게 유지되어, humidity가 애초에 설명력에 별다른 기여를 하지 못했던 변수였다는 것을 다시 확인할 수 있습니다.

**셀 6) train 예측값과 실제값의 피어슨 상관계수 (문제 1-2 정답)**

In [40]:
from scipy.stats import pearsonr

train_pred = model2.predict(X_train2)
corr, _ = pearsonr(train_pred, y_train)

print("1-2 정답:", round(corr, 3))

1-2 정답: 0.904


`model2.predict(X_train2)`로 재적합한 모형에 train 데이터를 그대로 다시 넣어 예측값을 구하고, 이 예측값과 실제 품질점수(y_train)의 상관관계를 `pearsonr()`로 계산했습니다.

**셀 7) test 데이터에 대한 예측값 계산**

In [41]:
X_test = test.drop(columns=['id', 'quality_score'])
y_test = test['quality_score']
X_test = sm.add_constant(X_test, has_constant='add')
X_test2 = X_test[sig_vars]

test_pred = model2.predict(X_test2)
print(test_pred.head())

0     92.578421
1     89.007466
2    114.701165
3     76.476355
4     82.794590
dtype: float64


test 데이터에도 상수항을 추가한 뒤, train과 동일한 컬럼 구성(sig_vars, humidity는 제외된 상태)으로 맞춰서 예측했습니다. `has_constant='add'` 옵션은 test 데이터에 우연히 이미 값이 1인 컬럼이 있을 가능성까지 고려해 상수항을 확실히 추가하기 위한 것입니다.

**셀 8) RMSE 계산 (문제 1-3 정답)**

In [42]:
import numpy as np

rmse = np.sqrt(np.mean((y_test - test_pred) ** 2))
print("1-3 정답:", round(rmse, 3))

1-3 정답: 5.855


`(y_test - test_pred) ** 2`로 각 배치마다 오차를 제곱하고, 평균을 낸 뒤 제곱근을 씌워 원래 단위(품질점수)로 되돌렸습니다. RMSE가 약 5.855라는 것은, test 데이터에서 모형의 예측이 실제 품질점수와 평균적으로 약 5.9점 정도 차이 난다는 뜻입니다.

### 전체 코드

In [43]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import pearsonr

df = pd.read_csv('process_quality.csv')

train = df[df['id'] <= 140].reset_index(drop=True)
test = df[df['id'] > 140].reset_index(drop=True)

# train으로 전체 변수 회귀모형 적합
X_train = train.drop(columns=['id', 'quality_score'])
y_train = train['quality_score']
X_train = sm.add_constant(X_train)
model = sm.OLS(y_train, X_train).fit()

# 1-1: 유의한 변수 개수
sig_vars = model.pvalues[model.pvalues < 0.05].index.tolist()
print("1-1 정답:", len(sig_vars) - 1)

# 유의한 변수만 재적합
X_train2 = X_train[sig_vars]
model2 = sm.OLS(y_train, X_train2).fit()

# 1-2: train 예측값-실제값 피어슨 상관계수
train_pred = model2.predict(X_train2)
corr, _ = pearsonr(train_pred, y_train)
print("1-2 정답:", round(corr, 3))

# test 예측 및 RMSE
X_test = test.drop(columns=['id', 'quality_score'])
y_test = test['quality_score']
X_test = sm.add_constant(X_test, has_constant='add')
X_test2 = X_test[sig_vars]
test_pred = model2.predict(X_test2)

# 1-3: RMSE
rmse = np.sqrt(np.mean((y_test - test_pred) ** 2))
print("1-3 정답:", round(rmse, 3))

1-1 정답: 3
1-2 정답: 0.904
1-3 정답: 5.855


### 정리

이 문제는 회귀모형을 train으로 학습하고 test로 평가하는 전체 흐름 안에서, 유의한 변수를 골라 재적합하는 과정과 두 가지 성능 지표(상관계수, RMSE)를 함께 다뤘습니다. train에서 구한 상관계수(0.904)가 test의 RMSE와 서로 다른 관점(패턴 유사도 vs 절대 오차 크기)의 정보를 준다는 점을 기억해두면 좋습니다.

### 🔍 실전 대비: 예상되는 변형과 접근 전략

기출 복원 자료에서 확인한 "다중회귀 + train/test 분할" 패턴은 실제로 여러 회차에서 반복되어온 대표적인 3유형 문제입니다. 다만 소문항 2, 3이 "train 상관계수 → test RMSE"가 아니라 "train R-squared → test 상관계수"처럼 지표의 조합이 바뀌어 나올 수 있습니다. 지표 이름과 그 지표를 어느 데이터(train인지 test인지)에 적용하는지를 정확히 구분해서 읽는 것이 핵심입니다.

**변형 예시: train이 아니라 test에서 상관계수를 구하라고 한다면**

In [44]:
test_corr, _ = pearsonr(test_pred, y_test)
print("test 예측값-실젯값 상관계수:", round(test_corr, 3))

test 예측값-실젯값 상관계수: 0.868


train 상관계수(0.904)와 test 상관계수(0.868)를 비교해보면, test에서 값이 더 낮아진 것을 확인할 수 있습니다. 이는 모형이 학습에 사용한 데이터(train)에서는 상대적으로 더 잘 들어맞고, 한 번도 보지 못한 데이터(test)에서는 그보다 조금 못한 성능을 보이는 자연스러운 현상입니다. 만약 반대로 test 상관계수가 train보다 눈에 띄게 높게 나온다면, 데이터 분할 과정에 문제가 없었는지 다시 점검해볼 필요가 있습니다.

---

## 문제 2. (10점)

제공된 데이터(subscription_churn.csv)는 구독형 서비스 이용 고객의 결제 및 이용 특성에 관한 자료이다. 월 이용료, 이용 개월 수, 모바일 앱 사용 여부, 프리미엄 지원 서비스 가입 여부 등에 따른 서비스 해지 여부(churn)를 예측하기 위해 로지스틱 회귀분석을 수행하고, 아래 소문항에 답하시오.

**제공 데이터 (subscription_churn.csv)**
| 컬럼명 | 설명 |
|---|---|
| id | 고객 번호 |
| monthly_fee | 월 이용료(원) |
| tenure_months | 이용 개월 수 |
| has_mobile_app | 모바일 앱 사용 여부 (0/1) |
| has_premium_support | 프리미엄 지원 서비스 가입 여부 (0/1) |
| churn | (종속변수) 서비스 해지 여부 — 0: 유지, 1: 해지 |

**문제 2-1** id를 제외한 모든 변수를 독립변수로 사용하여 로지스틱 회귀모형을 적합하시오. monthly_fee 변수의 유의확률(p-value)을 구하시오. (반올림하여 소수 셋째 자리까지 작성)

**문제 2-2** 위 모형을 바탕으로, 모바일 앱을 사용하지 않는 고객(0) 대비 사용하는 고객(1)의 해지 오즈비(Odds Ratio)를 구하시오. (반올림하여 소수 셋째 자리까지 작성)

**문제 2-3** 위 모형을 이용하여 각 고객의 해지 확률을 예측한 후, 예측된 해지 확률이 0.3을 초과하는 고객의 수를 구하시오.

### 문제 풀이 방향

이 문제는 train/test 분할 없이 전체 데이터를 그대로 사용해서 로지스틱 회귀모형을 적합하고, p-value 확인·오즈비 해석·확률 기반 인원수 세기까지 한 번에 다루는 종합 문제입니다. 재적합 없이 처음 적합한 모형을 그대로 사용한다는 점에서 지금까지의 로지스틱 회귀 문제들과 약간 다른 흐름입니다.

### 정답 코드 및 해설

**셀 1) 데이터 불러오기**

In [46]:
import pandas as pd

df = pd.read_csv('subscription_churn.csv')
print(df.shape)
df.head()

(950, 6)


,id,monthly_fee,tenure_months,has_mobile_app,has_premium_support,churn
0,1,32100.0,70,1,1,0
1,2,14500.0,51,0,0,0
2,3,49200.0,11,1,0,0
3,4,34900.0,9,1,1,1
4,5,49400.0,69,0,0,0


950명의 고객 데이터입니다. has_mobile_app과 has_premium_support는 이미 0/1로 인코딩되어 있어 별도의 문자열 변환 없이 바로 모델링에 사용할 수 있습니다.

**셀 2) 로지스틱 회귀모형 적합**

In [47]:
import statsmodels.api as sm

X = df.drop(columns=['id', 'churn'])
y = df['churn']
X = sm.add_constant(X)

model = sm.Logit(y, X).fit()
print(model.summary())

Optimization terminated successfully.
         Current function value: 0.525660
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                  churn   No. Observations:                  950
Model:                          Logit   Df Residuals:                      945
Method:                           MLE   Df Model:                            4
Date:                Sun, 19 Jul 2026   Pseudo R-squ.:                  0.1059
Time:                        22:32:13   Log-Likelihood:                -499.38
converged:                       True   LL-Null:                       -558.52
Covariance Type:            nonrobust   LLR p-value:                 1.243e-24
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  -0.5612      0.268     -2.097      0.036      -1.086      -0.037
mo

계수표를 보면 네 변수 모두 p-value가 0.05보다 작아 유의합니다. has_mobile_app과 has_premium_support의 계수가 음수라는 것은, 모바일 앱을 사용하거나 프리미엄 지원을 받는 고객일수록 해지 가능성이 낮아진다는 뜻입니다.

**셀 3) monthly_fee의 p-value 확인 (문제 2-1 정답)**

In [48]:
p_value = model.pvalues['monthly_fee']
print("monthly_fee p-value:", p_value)
print("2-1 정답:", round(p_value, 3))

monthly_fee p-value: 1.2065341302167804e-07
2-1 정답: 0.0


`model.pvalues['monthly_fee']`로 원하는 변수의 p-value만 바로 꺼냈습니다. 원래 값은 약 0.00000012로 매우 작은 수인데, 소수 셋째 자리까지 반올림하면 0.0으로 표시됩니다. 이렇게 반올림 결과가 0.0이 되더라도, 이는 "유의확률이 0이다"라는 뜻이 아니라 "유의수준 0.05보다 압도적으로 작다"는 뜻이므로, 실제로는 매우 강하게 유의한 변수라고 해석해야 합니다.

**셀 4) 모바일 앱 사용 여부에 따른 오즈비 (문제 2-2 정답)**

In [49]:
import numpy as np

odds_ratio = np.exp(model.params['has_mobile_app'])
print("2-2 정답:", round(odds_ratio, 3))

2-2 정답: 0.552


has_mobile_app의 계수(-0.5949)에 지수를 취해 오즈비를 구했습니다. 이 값이 1보다 작은 0.552라는 것은, 모바일 앱을 사용하는 고객(1)의 해지 오즈가 사용하지 않는 고객(0)의 오즈보다 낮다(약 0.552배, 즉 절반 가까이 낮다)는 뜻입니다. 오즈비가 1보다 크면 그 변수가 위험을 높이는 방향, 1보다 작으면 위험을 낮추는 방향이라는 것을 함께 기억해두면 해석이 쉬워집니다.

**셀 5) 전체 고객의 해지 확률 예측**

In [50]:
pred_prob = model.predict(X)
print(pred_prob.head())

0    0.053483
1    0.168045
2    0.561722
3    0.285027
4    0.286283
dtype: float64


이 문제는 train/test로 나누지 않고 전체 데이터에 그대로 모형을 적용해 확률을 예측했습니다. 새로운 데이터에 대한 일반화 성능을 확인하려는 목적이 아니라, "지금 가지고 있는 고객 전체 중 몇 명이 위험군인지"를 세는 것이 이 문제의 목적이기 때문입니다.

**셀 6) 확률 0.3 초과 고객 수 (문제 2-3 정답)**

In [51]:
count = (pred_prob > 0.3).sum()
print("2-3 정답:", count)

2-3 정답: 378


`pred_prob > 0.3`으로 각 고객의 예측 확률이 0.3을 초과하는지 True/False로 판별한 뒤, `.sum()`으로 True인 고객의 수를 세었습니다.

### 전체 코드

In [52]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

df = pd.read_csv('subscription_churn.csv')

X = df.drop(columns=['id', 'churn'])
y = df['churn']
X = sm.add_constant(X)

model = sm.Logit(y, X).fit()

# 2-1: monthly_fee의 p-value
print("2-1 정답:", round(model.pvalues['monthly_fee'], 3))

# 2-2: has_mobile_app 오즈비
odds_ratio = round(np.exp(model.params['has_mobile_app']), 3)
print("2-2 정답:", odds_ratio)

# 2-3: 확률 0.3 초과 고객 수
pred_prob = model.predict(X)
count = (pred_prob > 0.3).sum()
print("2-3 정답:", count)

Optimization terminated successfully.
         Current function value: 0.525660
         Iterations 6
2-1 정답: 0.0
2-2 정답: 0.552
2-3 정답: 378


### 정리

이 문제는 train/test 분할 없이 전체 데이터로 모형을 적합하고, p-value·오즈비·확률 기반 인원수라는 세 가지를 각각 다른 소문항에서 확인했습니다. p-value가 반올림 후 0.0으로 나오더라도 그 의미(유의수준보다 압도적으로 작다)를 정확히 해석하는 것과, 오즈비가 1보다 작을 때는 그 변수가 위험을 낮추는 방향이라는 것을 함께 기억해두면 좋습니다.

### 🔍 실전 대비: 예상되는 변형과 접근 전략

이 문제가 실제로 연습시키는 핵심은 **임계값(threshold)을 바꾸면 판단 결과가 어떻게 달라지는지에 대한 감각**입니다. 문제에서 제시하는 확률 기준값은 0.2, 0.3, 0.5 등으로 회차마다 다르게 나올 수 있으므로, 기준값이 바뀌어도 같은 코드 구조로 대응할 수 있어야 합니다.

**변형 예시: 임계값이 0.3이 아니라 0.5라면**

In [53]:
count_05 = (pred_prob > 0.5).sum()
print("임곗값 0.5 기준 인원:", count_05)

임곗값 0.5 기준 인원: 92


임계값을 0.3에서 0.5로 높이니 해당하는 고객 수가 378명에서 92명으로 크게 줄었습니다. 임계값이 높아질수록 "그 정도로 확실히 위험하다고 판단되는" 고객만 남기 때문에 당연히 인원수는 줄어듭니다. 이 관계(임계값이 높아지면 초과 인원수는 줄어든다)를 미리 알고 있으면, 계산 결과가 이 방향과 반대로 나왔을 때 코드에 실수가 있었다는 것을 스스로 점검할 수 있는 검산 기준으로 활용할 수 있습니다.

---

## 이번 회차에서 사용한 핵심 개념 정리
| 상황 | 함수/개념 | 핵심 이유 |
|---|---|---|
| id 기준 train/test 분할 | `df[df['id'] <= 기준]` / `df[df['id'] > 기준]` | 조건으로 행을 나누는 가장 기본적인 방법 |
| 특정 변수의 p-value만 조회 | `model.pvalues['변수명']` | summary 전체를 보지 않고 필요한 값만 바로 조회 |
| p-value가 반올림 후 0.0이 되는 경우 | 유의수준보다 압도적으로 작다는 의미 | "정확히 0이다"가 아니라 "매우 강하게 유의하다"로 해석 |
| 오즈비가 1보다 작을 때 | 그 변수가 해당 사건의 위험(오즈)을 낮추는 방향 | 1보다 크면 위험을 높이는 방향과 대비해서 이해 |
| 확률 임계값과 인원수의 관계 | 임계값이 높아질수록 초과 인원수는 감소 | 결과가 반대로 나오면 코드 점검 필요 |
| train/test 분할 없이 전체 데이터에 적용 | 목적에 따라 분할 여부가 달라짐 | 일반화 성능 확인이 아니라 현재 데이터의 현황 파악이 목적일 때는 분할 불필요 |